In [1]:
# UTILS
import numpy as np
from collections import defaultdict
import os
import pickle
import copy
import os
import numpy as np
import pandas as pd
import json
from collections import defaultdict
from sklearn.model_selection import train_test_split
import datetime
import time
import argparse
import pickle
import re
import random
import sys

# MODEL
import datetime
import math
import numpy as np
import torch
from torch import nn
from torch.nn import Module, Parameter
import torch.nn.functional as F
from collections import defaultdict

# MAIN
import argparse
import os
import sys
import time
import datetime
import torch

In [ ]:
import os
import pickle
import numpy as np
import random
import copy
from collections import defaultdict

def data_input_srgnn(opt):
    BASE_DIR = '/kaggle/input/PREPOCESSING/hasil_prepocessing'
    # BASE_DIR = './PREPOCESSING/hasil_prepocessing' 
    dataset_name='music4all'
    freq = 10  
    topu = 8000  
    PATH = 'music4all'
    n_music = 56047 + 1

    test_user_item_record = os.path.join(BASE_DIR, PATH, 'baseline_te_new_freq{}_partition8.lst'.format(freq))
    train_user_item_record = os.path.join(BASE_DIR, PATH, 'baseline_tr_freq{}_partition8.lst'.format(freq))
    
    test_user_item_file = open(test_user_item_record, 'rb')
    train_user_item_file = open(train_user_item_record, 'rb')

    test_lines = test_user_item_file.readlines()
    train_lines = train_user_item_file.readlines()
    
    train_user, train_music = [], []
    test_user, test_music = [], []

    usr2music_record_dic = defaultdict(set)

    flag = 1
    train_lenth,test_lenth=0,0
    user_set=set()
    
    for line in train_lines:
        if flag:
            flag = 0
            continue
        line = line.decode()
        user_id = int(line.split(',')[0])
        user_set.add(user_id)

        items_id = line.split(',')[1].split(':')
        int_items_id = list(map(int, items_id))  
        int_items_id_plus = np.array(int_items_id) + 1  
        train_lenth+=len(int_items_id)
        for item in int_items_id_plus:  
            usr2music_record_dic[user_id].add(item)

        train_user.append(user_id)
        train_music.append(int_items_id_plus.tolist())

    for line in test_lines:
        line = line.decode()
        user_id = int(line.split(',')[0])
        user_set.add(user_id)

        items_id = line.split(',')[1].split(':')

        int_items_id = list(map(int, items_id))  
        int_items_id_plus = np.array(int_items_id) + 1  
        test_lenth+=len(int_items_id)
        test_user.append(user_id)
        test_music.append(int_items_id_plus.tolist())

    test_user_item_file.close()
    train_user_item_file.close()

    train_data = train_user, train_music
    test_data = test_user, test_music

    train_data = data_split(train_data, usr2music_record_dic, opt, '', is_test=False)

    test_new_data = copy.deepcopy(test_data)
    test_new_data = data_split(test_new_data, usr2music_record_dic, opt, 'next-new-item', is_test=True)
    test_data = data_split(test_data, usr2music_record_dic, opt, 'next-one-item', is_test=True)

    return train_data, test_data, test_new_data

In [3]:
def build_graph(train_data):
    graph = nx.DiGraph()
    for seq in train_data:
        for i in range(len(seq) - 1):
            if graph.get_edge_data(seq[i], seq[i + 1]) is None:
                weight = 1
            else:
                weight = graph.get_edge_data(seq[i], seq[i + 1])['weight'] + 1
            graph.add_edge(seq[i], seq[i + 1], weight=weight)
    for node in graph.nodes:
        sum = 0
        for j, i in graph.in_edges(node):
            sum += graph.get_edge_data(j, i)['weight']
        if sum != 0:
            for j, i in graph.in_edges(i):
                graph.add_edge(j, i, weight=graph.get_edge_data(j, i)['weight'] / sum)
    return graph

In [4]:
def data_split(data, usr2music_record_dic, opt, mode, is_test=False):
    windowLenth = opt.windowLenth
    data_size = opt.data_size
    step = opt.slide_step  

    inputs, targets = [], []

    if is_test == False:
        for user_id, music_slice in zip(data[0], data[1]):
            for i in range(windowLenth, int(len(music_slice)), step):
                inputs.append(music_slice[i - windowLenth:i])
                targets.append(music_slice[i])
    else:
        if mode == 'next-new-item':
            for user_id, music_slice in zip(data[0], data[1]):
                for i in range(windowLenth, int(len(music_slice))):
                    t = music_slice[i]
                    if t in usr2music_record_dic[user_id] or t in music_slice[i - windowLenth:i]:
                        continue
                    inputs.append(music_slice[i - windowLenth:i])
                    targets.append(music_slice[i])

        elif mode == 'next-recent-new-item':
            for user_id, music_slice in zip(data[0], data[1]):
                for i in range(windowLenth, int(len(music_slice))):
                    t = music_slice[i]
                    if t in music_slice[i - windowLenth:i]:
                        continue
                    inputs.append(music_slice[i - windowLenth:i])
                    targets.append(music_slice[i])

        elif mode == 'next-one-item':
            for user_id, music_slice in zip(data[0], data[1]):
                for i in range(windowLenth, int(len(music_slice))):
                    inputs.append(music_slice[i - windowLenth:i])
                    targets.append(music_slice[i])

    return inputs, targets

In [5]:
def data_masks(all_usr_pois, item_tail):
    us_lens = [len(upois) for upois in all_usr_pois]
    len_max = max(us_lens)
    us_pois = [upois + item_tail * (len_max - le) for upois, le in zip(all_usr_pois, us_lens)]
    us_msks = [[1] * le + [0] * (len_max - le) for le in us_lens]
    return us_pois, us_msks, len_max

In [6]:
class Data():
    def __init__(self, data, opt, shuffle=False, graph=None):
        inputs = data[0]
        inputs, mask, len_max = data_masks(inputs, [0])
        self.inputs = np.asarray(inputs)
        self.mask = np.asarray(mask)
        self.len_max = len_max
        self.targets = np.asarray(data[1])
        self.length = len(inputs)
        self.shuffle = shuffle
        self.windowLenth = opt.windowLenth
        self.graph = graph

    def generate_batch(self, batch_size):
        if self.shuffle:
            shuffled_arg = np.arange(self.length)
            np.random.shuffle(shuffled_arg)
            self.inputs = self.inputs[shuffled_arg]
            self.mask = self.mask[shuffled_arg]
            self.targets = self.targets[shuffled_arg]
        n_batch = int(self.length / batch_size)
        if self.length % batch_size != 0:
            n_batch += 1
        slices = np.split(np.arange(n_batch * batch_size), n_batch)
        slices[-1] = slices[-1][:(self.length - batch_size * (n_batch - 1))]
        return slices

    def get_slice(self, i):
        inputs, mask, targets = self.inputs[i], self.mask[i], self.targets[i]
        items, n_node, A, alias_inputs = [], [], [], []
        for u_input in inputs:
            n_node.append(len(np.unique(u_input)))
        max_n_node = np.max(n_node)
        for u_input in inputs:
            node = np.unique(u_input)
            items.append(node.tolist() + (max_n_node - len(node)) * [0])
            u_A = np.zeros((max_n_node, max_n_node))
            for j in np.arange(len(u_input) - 1):
                if u_input[j + 1] == 0:
                    break
                u = np.where(node == u_input[j])[0][0]
                v = np.where(node == u_input[j + 1])[0][0]
                u_A[u][v] = 1
            u_sum_in = np.sum(u_A, 0)
            u_sum_in[np.where(u_sum_in == 0)] = 1
            u_A_in = np.divide(u_A, u_sum_in)
            u_sum_out = np.sum(u_A, 1)
            u_sum_out[np.where(u_sum_out == 0)] = 1
            u_A_out = np.divide(u_A.transpose(), u_sum_out)
            u_A = np.concatenate([u_A_in, u_A_out]).transpose()
            A.append(u_A)
            alias_inputs.append([np.where(node == j)[0][0] for j in u_input])
        return alias_inputs, A, items, mask, targets

# MODEL

In [7]:
class GNN(Module):
    def __init__(self, hidden_size, step=1):
        super(GNN, self).__init__()
        self.step = step
        self.hidden_size = hidden_size
        self.input_size = hidden_size * 2
        self.gate_size = 3 * hidden_size
        self.w_ih = Parameter(torch.Tensor(self.gate_size, self.input_size))
        self.w_hh = Parameter(torch.Tensor(self.gate_size, self.hidden_size))
        self.b_ih = Parameter(torch.Tensor(self.gate_size))
        self.b_hh = Parameter(torch.Tensor(self.gate_size))
        self.b_iah = Parameter(torch.Tensor(self.hidden_size))
        self.b_oah = Parameter(torch.Tensor(self.hidden_size))

        self.linear_edge_in = nn.Linear(self.hidden_size, self.hidden_size, bias=True)
        self.linear_edge_out = nn.Linear(self.hidden_size, self.hidden_size, bias=True)
        self.linear_edge_f = nn.Linear(self.hidden_size, self.hidden_size, bias=True)

    def GNNCell(self, A, hidden):
        input_in = torch.matmul(A[:, :, :A.shape[1]], self.linear_edge_in(hidden)) + self.b_iah
        input_out = torch.matmul(A[:, :, A.shape[1]: 2 * A.shape[1]], self.linear_edge_out(hidden)) + self.b_oah
        inputs = torch.cat([input_in, input_out], 2)
        gi = F.linear(inputs, self.w_ih, self.b_ih)
        gh = F.linear(hidden, self.w_hh, self.b_hh)
        i_r, i_i, i_n = gi.chunk(3, 2)
        h_r, h_i, h_n = gh.chunk(3, 2)
        resetgate = torch.sigmoid(i_r + h_r)
        inputgate = torch.sigmoid(i_i + h_i)
        newgate = torch.tanh(i_n + resetgate * h_n)
        hy = newgate + inputgate * (hidden - newgate)
        return hy

    def forward(self, A, hidden):
        for i in range(self.step):
            hidden = self.GNNCell(A, hidden)
        return hidden

In [ ]:
class SessionGraph(Module):
    def __init__(self, opt, n_music):
        super(SessionGraph, self).__init__()
        self.hidden_size = opt.hiddenSize
        self.n_music = n_music
        self.batch_size = opt.batchSize
        self.embedding = nn.Embedding(self.n_music, self.hidden_size)
        self.gnn = GNN(self.hidden_size, step=opt.step)
        self.linear_one = nn.Linear(self.hidden_size, self.hidden_size, bias=True)
        self.linear_two = nn.Linear(self.hidden_size, self.hidden_size, bias=True)
        self.linear_three = nn.Linear(self.hidden_size, 1, bias=False)
        self.linear_transform = nn.Linear(self.hidden_size * 2, self.hidden_size, bias=True)
        self.loss_function = nn.CrossEntropyLoss()
        self.optimizer = torch.optim.Adam(self.parameters(), lr=opt.lr, weight_decay=opt.l2)
        self.scheduler = torch.optim.lr_scheduler.StepLR(self.optimizer, step_size=opt.lr_dc_step, gamma=opt.lr_dc)
        self.reset_parameters()

    def reset_parameters(self):
        stdv = 1.0 / math.sqrt(self.hidden_size)
        for weight in self.parameters():
            weight.data.uniform_(-stdv, stdv)

    def compute_scores(self, hidden, mask):
        ht = hidden[torch.arange(mask.shape[0]).long(), torch.sum(mask, 1) - 1]  
        q1 = self.linear_one(ht).view(ht.shape[0], 1, ht.shape[1])  
        q2 = self.linear_two(hidden)  
        alpha = self.linear_three(torch.sigmoid(q1 + q2))
        a = torch.sum(alpha * hidden * mask.view(mask.shape[0], -1, 1).float(), 1)
        a = self.linear_transform(torch.cat([a, ht], 1))
        b = self.embedding.weight[1:]  
        scores = torch.matmul(a, b.transpose(1, 0))
        return scores

    def forward(self, inputs, A):
        hidden = self.embedding(inputs)
        hidden = self.gnn(A, hidden)
        return hidden

In [9]:
def trans_to_cuda(variable):
    if torch.cuda.is_available():
        return variable.cuda()
    else:
        return variable

In [10]:
def trans_to_cpu(variable):
    if torch.cuda.is_available():
        return variable.cpu()
    else:
        return variable

In [11]:
def forward(model, i, data):
    alias_inputs, A, items, mask, targets = data.get_slice(i)
    alias_inputs = trans_to_cuda(torch.Tensor(alias_inputs).long())
    items = trans_to_cuda(torch.Tensor(items).long())
    A = np.array(A)
    A = trans_to_cuda(torch.Tensor(A).float())
    mask = trans_to_cuda(torch.Tensor(mask).long())
    hidden = model(items, A)
    get = lambda i: hidden[i][alias_inputs[i]]
    seq_hidden = torch.stack([get(i) for i in torch.arange(len(alias_inputs)).long()])
    return targets, model.compute_scores(seq_hidden, mask)

In [12]:
def predict(model, test_data):
    N = 21
    hit_res = np.zeros(N)
    mrr_res = np.zeros(N)

    model.eval()
    hit = defaultdict(list)
    mrr = defaultdict(list)
    slices = test_data.generate_batch(model.batch_size)
    for i in slices:
        targets, scores = forward(model, i, test_data)
        sub_scores = scores.topk(N - 1)[1]
        sub_scores = trans_to_cpu(sub_scores).detach().numpy()

        for score, target in zip(sub_scores, targets):
            target = target - 1
            for topN in range(1, N):
                topN_item = score[:topN]
                hit[topN].append(np.isin(target, topN_item))
                common = []
                if target in topN_item:
                    common.append(target)
                    mrr[topN].append(1 / (np.where(topN_item == target)[0][0] + 1))
                else:
                    mrr[topN].append(0)

    for topN in range(1, N):
        hit_res[topN] = np.mean(hit[topN]) * 100
        mrr_res[topN] = np.mean(mrr[topN]) * 100

    print("hit:\n{}".format(hit_res[1:]))
    print("mrr:\n{}".format(mrr_res[1:]))

    return hit_res[1:], mrr_res[1:]

In [13]:
def train_test(model, train_data, test_data, test_new_data):
    print('start training: ')
    model.scheduler.step()
    model.train()
    total_loss = 0.0
    slices = train_data.generate_batch(model.batch_size)
    for i, j in zip(slices, np.arange(len(slices))):
        model.optimizer.zero_grad()
        targets, scores = forward(model, i, train_data)
        targets = trans_to_cuda(torch.Tensor(targets).long())
        loss = model.loss_function(scores, targets-1)
        loss.backward()
        model.optimizer.step()
        total_loss += loss
        if j % int(len(slices) / 5 + 1) == 0:
            print('[%d/%d] Loss: %.4f' % (j, len(slices), loss.item()))
    print('\tLoss:\t%.3f' % total_loss)

    print('start predicting (next-one): ')
    hit_next_one, mrr_next_one = predict(model, test_data)

    print('start predicting (next-new): ')
    hit_next_new, mrr_next_new = predict(model, test_new_data)

    return (hit_next_one, mrr_next_one), (hit_next_new, mrr_next_new)

# MAIN

In [14]:
SEED = 42
sys.setrecursionlimit(100000)
torch.manual_seed(1000)
torch.backends.cudnn.deterministic = True

parser = argparse.ArgumentParser()

parser.add_argument('--dataset', default='music4all', help='dataset name: music4all')
parser.add_argument('--detail', default='Baseline(SRGNN)', help='Deskripsi kode untuk pembeda')
parser.add_argument('--data_size', default=1, type=float, help='Bagian data yang digunakan')
parser.add_argument('--slide_step', default=1, type=int, help='Langkah sliding window')
parser.add_argument('--windowLenth', type=int, default=3, help='Panjang maksimum sliding window')
parser.add_argument('--batchSize', type=int, default=256, help='Ukuran batch input')
parser.add_argument('--hiddenSize', type=int, default=100, help='Ukuran hidden state')
parser.add_argument('--epoch', type=int, default=10, help='Jumlah epoch untuk training')
parser.add_argument('--lr', type=float, default=0.001, help='Learning rate')
parser.add_argument('--lr_dc', type=float, default=0.1, help='Tingkat penurunan learning rate')
parser.add_argument('--lr_dc_step', type=int, default=3, help='Jumlah langkah sebelum learning rate turun')
parser.add_argument('--l2', type=float, default=1e-5, help='L2 penalty')
parser.add_argument('--step', type=int, default=1, help='Jumlah propagasi GNN')

opt = parser.parse_args(args=[])

# Checkpoint configuration untuk Kaggle
CHECKPOINT_INPUT_DIR = '/kaggle/input/checkpoint/pytorch/default/1'
CHECKPOINT_OUTPUT_DIR = '/kaggle/working/'
SAVE_EVERY = 5  # Save checkpoint every 5 epochs

start_time = time.time()

def save_checkpoint(model, epoch, results_next_one, results_next_new):
    """Simpan checkpoint model ke /kaggle/working"""
    os.makedirs(CHECKPOINT_OUTPUT_DIR, exist_ok=True)
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': model.optimizer.state_dict(),
        'scheduler_state_dict': model.scheduler.state_dict(),
        'results_next_one': results_next_one,
        'results_next_new': results_next_new,
        'opt': opt
    }
    checkpoint_path = os.path.join(CHECKPOINT_OUTPUT_DIR, f'baselinemodel_checkpoint_epoch_{epoch + 1}.pt')
    torch.save(checkpoint, checkpoint_path)
    print(f"Checkpoint saved: {checkpoint_path}")
    return checkpoint_path

def find_checkpoint_in_input():
    """Cari checkpoint di /kaggle/input/checkpoint/pytorch/default/1"""
    checkpoint_files = []
    
    if os.path.exists(CHECKPOINT_INPUT_DIR):
        print(f"Scanning checkpoint input directory: {CHECKPOINT_INPUT_DIR}")
        files = os.listdir(CHECKPOINT_INPUT_DIR)
        print(f"Files found: {files}")
        
        for file in files:
            if file.startswith('baselinemodel_checkpoint_epoch_') and file.endswith('.pt'):
                full_path = os.path.join(CHECKPOINT_INPUT_DIR, file)
                try:
                    epoch_num = int(file.split('_epoch_')[1].split('.pt')[0])
                    checkpoint_files.append((epoch_num, full_path))
                    print(f"Found checkpoint: {file} (epoch {epoch_num})")
                except:
                    continue
    else:
        print(f"Checkpoint input directory not found: {CHECKPOINT_INPUT_DIR}")
    
    return checkpoint_files

def find_checkpoint_in_working():
    """Cari checkpoint di /kaggle/working dari session sebelumnya"""
    checkpoint_files = []
    
    if os.path.exists(CHECKPOINT_OUTPUT_DIR):
        files = os.listdir(CHECKPOINT_OUTPUT_DIR)
        for file in files:
            if file.startswith('baselinemodel_checkpoint_epoch_') and file.endswith('.pt'):
                full_path = os.path.join(CHECKPOINT_OUTPUT_DIR, file)
                try:
                    epoch_num = int(file.split('_epoch_')[1].split('.pt')[0])
                    checkpoint_files.append((epoch_num, full_path))
                except:
                    continue
    
    return checkpoint_files

def load_latest_checkpoint(model):
    """Load checkpoint terbaru dari input atau working directory"""
    print("Checking for existing checkpoints...")
    
    # Cari checkpoint di kedua lokasi
    input_checkpoints = find_checkpoint_in_input()
    working_checkpoints = find_checkpoint_in_working()
    
    # Gabungkan dan cari yang terbaru
    all_checkpoints = input_checkpoints + working_checkpoints
    
    if not all_checkpoints:
        print("No checkpoint found. Starting from scratch.")
        return 0, [], []
    
    # Ambil checkpoint dengan epoch tertinggi
    latest_epoch, latest_path = max(all_checkpoints, key=lambda x: x[0])
    
    print(f"Found latest checkpoint: {latest_path} (epoch {latest_epoch})")
    try:
        from argparse import Namespace
        import torch.serialization
        torch.serialization.add_safe_globals([Namespace])
        checkpoint = torch.load(latest_path, map_location='cpu', weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        model.scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        results_next_one = checkpoint.get('results_next_one', [])
        results_next_new = checkpoint.get('results_next_new', [])
        print(f"Resuming training from epoch {start_epoch}")
        return start_epoch, results_next_one, results_next_new
    except Exception as e:
        print(f"Error loading checkpoint: {e}")
        print("Starting training from scratch")
        return 0, [], []

def main():
    print("Model Baseline (SRGNN)")
    print(f"Checkpoint input directory: {CHECKPOINT_INPUT_DIR}")
    print(f"Checkpoint output directory: {CHECKPOINT_OUTPUT_DIR}")

    opt.dataset = 'music4all'
    print(f'============= dataset: {opt.dataset} =============')

    n_music = 56047 + 1

    print("--------------- Data Input ---------------")
    train_data_raw, test_data_raw, test_new_data_raw = data_input_srgnn(opt)

    print(f"Total split train records: {len(train_data_raw[0])}")
    print(f"Total split test records (next-one): {len(test_data_raw[0])}")
    print(f"Total split test records (next-new): {len(test_new_data_raw[0])}")

    train_data = Data(train_data_raw, opt)
    test_data = Data(test_data_raw, opt)
    test_new_data = Data(test_new_data_raw, opt)

    model = trans_to_cuda(SessionGraph(opt, n_music))

    # Auto-load checkpoint jika ada
    start_epoch, results_next_one, results_next_new = load_latest_checkpoint(model)
    
    if start_epoch >= opt.epoch:
        print(f"Training already completed (epoch {start_epoch} >= {opt.epoch})")
        print("To continue training, increase --epoch parameter")
        return

    best_hit_next_one = None
    best_mrr_next_one = None
    best_hit_next_new = None
    best_mrr_next_new = None

    # Training loop dengan checkpoint otomatis
    for epoch in range(start_epoch, opt.epoch):
        print('-------------------------------------------------------')
        print(f'epoch: {epoch}')
        
        try:
            (hit_next_one, mrr_next_one), (hit_next_new, mrr_next_new) = train_test(model, train_data, test_data, test_new_data)
            sys.stdout.flush()

            best_hit_next_one = hit_next_one
            best_mrr_next_one = mrr_next_one
            best_hit_next_new = hit_next_new
            best_mrr_next_new = mrr_next_new

            result_one = {
                'epoch': epoch + 1,
                'hitrate10': float(best_hit_next_one[9]),
                'hitrate20': float(best_hit_next_one[19]),
                'mrr10': float(best_mrr_next_one[9]),
                'mrr20': float(best_mrr_next_one[19]),
            }
            result_new = {
                'epoch': epoch + 1,
                'hitrate10': float(best_hit_next_new[9]),
                'hitrate20': float(best_hit_next_new[19]),
                'mrr10': float(best_mrr_next_new[9]),
                'mrr20': float(best_mrr_next_new[19]),
            }

            results_next_one.append(result_one)
            results_next_new.append(result_new)

            # Auto-save checkpoint setiap N epoch
            if (epoch + 1) % SAVE_EVERY == 0:
                save_checkpoint(model, epoch, results_next_one, results_next_new)
                print(f"Auto-saved checkpoint at epoch {epoch + 1}")

        except Exception as e:
            print(f"Error during training at epoch {epoch}: {e}")
            # Simpan emergency checkpoint
            emergency_path = save_checkpoint(model, epoch - 1, results_next_one, results_next_new)
            print(f"Emergency checkpoint saved: {emergency_path}")
            raise e

    # Simpan final checkpoint jika training selesai
    if opt.epoch > 0:
        final_path = save_checkpoint(model, opt.epoch - 1, results_next_one, results_next_new)
        print(f"Final checkpoint saved: {final_path}")

    del model

    end_time = time.time()
    duration_hours = (end_time - start_time) / 3600
    print(f"\nTotal waktu yang dihabiskan: {duration_hours:.2f} jam")

    results_one_df = pd.DataFrame(results_next_one)
    results_new_df = pd.DataFrame(results_next_new)

    # Simpan hasil ke /kaggle/working dengan timestamp
    results_one_path = os.path.join(CHECKPOINT_OUTPUT_DIR, f'results_baselinemodel_next_one.csv')
    results_new_path = os.path.join(CHECKPOINT_OUTPUT_DIR, f'results_baselinemodel_next_new.csv')

    results_one_df.to_csv(results_one_path, index=False)
    results_new_df.to_csv(results_new_path, index=False)

    print(f"Saved results to:\n  {results_one_path}\n  {results_new_path}")

if __name__ == '__main__':
    main()

Model Baseline (SRGNN)
Checkpoint input directory: /kaggle/input/checkpoint/pytorch/default/1
Checkpoint output directory: /kaggle/working/
============= dataset: music4all =============
--------------- Data Input ---------------
Total split train records: 2471719
Total split test records (next-one): 603960
Total split test records (next-new): 272431
Checking for existing checkpoints...
Checkpoint input directory not found: /kaggle/input/checkpoint/pytorch/default/1
No checkpoint found. Starting from scratch.
-------------------------------------------------------
epoch: 0
start training: 


/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:227: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


[0/9656] Loss: 10.9395
[1932/9656] Loss: 8.6984
[3864/9656] Loss: 8.4871
[5796/9656] Loss: 4.0565
[7728/9656] Loss: 9.4921
	Loss:	70074.047
start predicting (next-one): 
hit:
[14.30376184 20.38876747 24.4123783  27.17415061 29.27313067 30.9894695
 32.45281144 33.70918604 34.82432611 35.79160872 36.64199616 37.34518842
 37.95681833 38.50834492 39.00423869 39.4726472  39.90347043 40.33677727
 40.72438572 41.12524008]
mrr:
[14.30376184 17.34626465 18.68746827 19.37791134 19.79770735 20.08376383
 20.29281267 20.4498595  20.57376395 20.67049221 20.74780016 20.80639952
 20.85344797 20.89284273 20.92590231 20.95517784 20.98052039 21.00459299
 21.02499343 21.04503615]
start predicting (next-new): 
hit:
[ 7.28404624 12.00524169 15.34113225 17.61143189 19.36380221 20.8118753
 22.04337979 23.07997254 23.929729   24.73286814 25.47140377 26.11743891
 26.69409869 27.21716692 27.67709989 28.1249197  28.530527   28.92769178
 29.29328894 29.69485851]
mrr:
[ 7.28404624  9.64464396 10.75660748 11.3241823

In [16]:
SEED = 42
sys.setrecursionlimit(100000)
torch.manual_seed(1000)
torch.backends.cudnn.deterministic = True

parser = argparse.ArgumentParser()

parser.add_argument('--dataset', default='music4all', help='dataset name: music4all')
parser.add_argument('--detail', default='Baseline(SRGNN)', help='Deskripsi kode untuk pembeda')
parser.add_argument('--data_size', default=1, type=float, help='Bagian data yang digunakan')
parser.add_argument('--slide_step', default=1, type=int, help='Langkah sliding window')
parser.add_argument('--windowLenth', type=int, default=3, help='Panjang maksimum sliding window')
parser.add_argument('--batchSize', type=int, default=256, help='Ukuran batch input')
parser.add_argument('--hiddenSize', type=int, default=100, help='Ukuran hidden state')
parser.add_argument('--epoch', type=int, default=20, help='Jumlah epoch untuk training')
parser.add_argument('--lr', type=float, default=0.001, help='Learning rate')
parser.add_argument('--lr_dc', type=float, default=0.1, help='Tingkat penurunan learning rate')
parser.add_argument('--lr_dc_step', type=int, default=3, help='Jumlah langkah sebelum learning rate turun')
parser.add_argument('--l2', type=float, default=1e-5, help='L2 penalty')
parser.add_argument('--step', type=int, default=1, help='Jumlah propagasi GNN')

opt = parser.parse_args(args=[])

# Checkpoint configuration untuk Kaggle
CHECKPOINT_INPUT_DIR = '/kaggle/input/checkpoint_baselinemodel_epoch10/pytorch/default/1'
CHECKPOINT_OUTPUT_DIR = '/kaggle/working/'
SAVE_EVERY = 5  # Save checkpoint every 5 epochs

start_time = time.time()

def save_checkpoint(model, epoch, results_next_one, results_next_new):
    """Simpan checkpoint model ke /kaggle/working"""
    os.makedirs(CHECKPOINT_OUTPUT_DIR, exist_ok=True)
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': model.optimizer.state_dict(),
        'scheduler_state_dict': model.scheduler.state_dict(),
        'results_next_one': results_next_one,
        'results_next_new': results_next_new,
        'opt': opt
    }
    checkpoint_path = os.path.join(CHECKPOINT_OUTPUT_DIR, f'baselinemodel_checkpoint_epoch_{epoch + 1}.pt')
    torch.save(checkpoint, checkpoint_path)
    print(f"Checkpoint saved: {checkpoint_path}")
    return checkpoint_path

def find_checkpoint_in_input():
    """Cari checkpoint di /kaggle/input/checkpoint/pytorch/default/1"""
    checkpoint_files = []
    
    if os.path.exists(CHECKPOINT_INPUT_DIR):
        print(f"Scanning checkpoint input directory: {CHECKPOINT_INPUT_DIR}")
        files = os.listdir(CHECKPOINT_INPUT_DIR)
        print(f"Files found: {files}")
        
        for file in files:
            if file.startswith('baselinemodel_checkpoint_epoch_') and file.endswith('.pt'):
                full_path = os.path.join(CHECKPOINT_INPUT_DIR, file)
                try:
                    epoch_num = int(file.split('_epoch_')[1].split('.pt')[0])
                    checkpoint_files.append((epoch_num, full_path))
                    print(f"Found checkpoint: {file} (epoch {epoch_num})")
                except:
                    continue
    else:
        print(f"Checkpoint input directory not found: {CHECKPOINT_INPUT_DIR}")
    
    return checkpoint_files

def find_checkpoint_in_working():
    """Cari checkpoint di /kaggle/working dari session sebelumnya"""
    checkpoint_files = []
    
    if os.path.exists(CHECKPOINT_OUTPUT_DIR):
        files = os.listdir(CHECKPOINT_OUTPUT_DIR)
        for file in files:
            if file.startswith('baselinemodel_checkpoint_epoch_') and file.endswith('.pt'):
                full_path = os.path.join(CHECKPOINT_OUTPUT_DIR, file)
                try:
                    epoch_num = int(file.split('_epoch_')[1].split('.pt')[0])
                    checkpoint_files.append((epoch_num, full_path))
                except:
                    continue
    
    return checkpoint_files

def load_latest_checkpoint(model):
    """Load checkpoint terbaru dari input atau working directory"""
    print("Checking for existing checkpoints...")
    
    # Cari checkpoint di kedua lokasi
    input_checkpoints = find_checkpoint_in_input()
    working_checkpoints = find_checkpoint_in_working()
    
    # Gabungkan dan cari yang terbaru
    all_checkpoints = input_checkpoints + working_checkpoints
    
    if not all_checkpoints:
        print("No checkpoint found. Starting from scratch.")
        return 0, [], []
    
    # Ambil checkpoint dengan epoch tertinggi
    latest_epoch, latest_path = max(all_checkpoints, key=lambda x: x[0])
    
    print(f"Found latest checkpoint: {latest_path} (epoch {latest_epoch})")
    try:
        from argparse import Namespace
        import torch.serialization
        torch.serialization.add_safe_globals([Namespace])
        checkpoint = torch.load(latest_path, map_location='cpu', weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        model.scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        results_next_one = checkpoint.get('results_next_one', [])
        results_next_new = checkpoint.get('results_next_new', [])
        print(f"Resuming training from epoch {start_epoch}")
        return start_epoch, results_next_one, results_next_new
    except Exception as e:
        print(f"Error loading checkpoint: {e}")
        print("Starting training from scratch")
        return 0, [], []

def main():
    print("Resume Model Baseline (SRGNN)")
    print(f"Checkpoint input directory: {CHECKPOINT_INPUT_DIR}")
    print(f"Checkpoint output directory: {CHECKPOINT_OUTPUT_DIR}")

    opt.dataset = 'music4all'
    print(f'============= dataset: {opt.dataset} =============')

    n_music = 56047 + 1

    print("--------------- Data Input ---------------")
    train_data_raw, test_data_raw, test_new_data_raw = data_input_srgnn(opt)

    print(f"Total split train records: {len(train_data_raw[0])}")
    print(f"Total split test records (next-one): {len(test_data_raw[0])}")
    print(f"Total split test records (next-new): {len(test_new_data_raw[0])}")

    train_data = Data(train_data_raw, opt)
    test_data = Data(test_data_raw, opt)
    test_new_data = Data(test_new_data_raw, opt)

    model = trans_to_cuda(SessionGraph(opt, n_music))

    # Auto-load checkpoint jika ada
    start_epoch, results_next_one, results_next_new = load_latest_checkpoint(model)
    
    if start_epoch >= opt.epoch:
        print(f"Training already completed (epoch {start_epoch} >= {opt.epoch})")
        print("To continue training, increase --epoch parameter")
        return

    best_hit_next_one = None
    best_mrr_next_one = None
    best_hit_next_new = None
    best_mrr_next_new = None

    # Training loop dengan checkpoint otomatis
    for epoch in range(start_epoch, opt.epoch):
        print('-------------------------------------------------------')
        print(f'epoch: {epoch}')
        
        try:
            (hit_next_one, mrr_next_one), (hit_next_new, mrr_next_new) = train_test(model, train_data, test_data, test_new_data)
            sys.stdout.flush()

            best_hit_next_one = hit_next_one
            best_mrr_next_one = mrr_next_one
            best_hit_next_new = hit_next_new
            best_mrr_next_new = mrr_next_new

            result_one = {
                'epoch': epoch + 1,
                'hitrate10': float(best_hit_next_one[9]),
                'hitrate20': float(best_hit_next_one[19]),
                'mrr10': float(best_mrr_next_one[9]),
                'mrr20': float(best_mrr_next_one[19]),
            }
            result_new = {
                'epoch': epoch + 1,
                'hitrate10': float(best_hit_next_new[9]),
                'hitrate20': float(best_hit_next_new[19]),
                'mrr10': float(best_mrr_next_new[9]),
                'mrr20': float(best_mrr_next_new[19]),
            }

            results_next_one.append(result_one)
            results_next_new.append(result_new)

            # Auto-save checkpoint setiap N epoch
            if (epoch + 1) % SAVE_EVERY == 0:
                save_checkpoint(model, epoch, results_next_one, results_next_new)
                print(f"Auto-saved checkpoint at epoch {epoch + 1}")

        except Exception as e:
            print(f"Error during training at epoch {epoch}: {e}")
            # Simpan emergency checkpoint
            emergency_path = save_checkpoint(model, epoch - 1, results_next_one, results_next_new)
            print(f"Emergency checkpoint saved: {emergency_path}")
            raise e

    # Simpan final checkpoint jika training selesai
    if opt.epoch > 0:
        final_path = save_checkpoint(model, opt.epoch - 1, results_next_one, results_next_new)
        print(f"Final checkpoint saved: {final_path}")

    del model

    end_time = time.time()
    duration_hours = (end_time - start_time) / 3600
    print(f"\nTotal waktu yang dihabiskan: {duration_hours:.2f} jam")

    results_one_df = pd.DataFrame(results_next_one)
    results_new_df = pd.DataFrame(results_next_new)

    # Simpan hasil ke /kaggle/working dengan timestamp
    results_one_path = os.path.join(CHECKPOINT_OUTPUT_DIR, f'results_baselinemodel_next_one.csv')
    results_new_path = os.path.join(CHECKPOINT_OUTPUT_DIR, f'results_baselinemodel_next_new.csv')

    results_one_df.to_csv(results_one_path, index=False)
    results_new_df.to_csv(results_new_path, index=False)

    print(f"Saved results to:\n  {results_one_path}\n  {results_new_path}")

if __name__ == '__main__':
    main()

Resume Model Baseline (SRGNN)
Checkpoint input directory: /kaggle/input/checkpoint_baselinemodel_epoch10/pytorch/default/1
Checkpoint output directory: /kaggle/working/
============= dataset: music4all =============
--------------- Data Input ---------------
Total split train records: 2471719
Total split test records (next-one): 603960
Total split test records (next-new): 272431
Checking for existing checkpoints...
Scanning checkpoint input directory: /kaggle/input/checkpoint_baselinemodel_epoch10/pytorch/default/1
Files found: ['baselinemodel_checkpoint_epoch_10.pt']
Found checkpoint: baselinemodel_checkpoint_epoch_10.pt (epoch 10)
Found latest checkpoint: /kaggle/input/checkpoint_baselinemodel_epoch10/pytorch/default/1/baselinemodel_checkpoint_epoch_10.pt (epoch 10)
Resuming training from epoch 10
-------------------------------------------------------
epoch: 10
start training: 
[0/9656] Loss: 6.1569
[1932/9656] Loss: 6.4154
[3864/9656] Loss: 6.9435
[5796/9656] Loss: 3.6586
[7728/965